## 🛠️ Pipeline Activities

When building a pipeline in the ADF Authoring canvas, **Activities** represent the individual processing steps or actions you want to execute. They are grouped into several logical categories:

1. **Move and Transform**:
   * *What it is*: Activities used to ingest, copy, and transform data at scale.
   * *Key Examples*: **Copy Activity** (for moving data) and **Mapping Data Flows** (for code-free transformations).

2. **Synapse**:
   * *What it is*: Activities designed to interact directly with Azure Synapse Analytics workspaces.
   * *Key Use Case*: Triggering Synapse SQL scripts or Spark notebooks directly from your ADF pipeline.

3. **Azure Data Explorer**:
   * *What it is*: Activities used to run control or data commands against Azure Data Explorer (Kusto) clusters for real-time log and telemetry analytics.

4. **Azure Function**:
   * *What it is*: Allows you to invoke custom serverless code written in an Azure Function (Python, C#, JavaScript) as a step inside your data pipeline.

5. **Batch Service**:
   * *What it is*: Runs custom parallel and high-performance computing (HPC) batch jobs on managed Azure compute pools.

6. **Databricks**:
   * *What it is*: Integrates Apache Spark workloads by allowing you to execute Notebooks, JARs, or Python scripts directly within your ADF pipeline.

7. **General**:
   * *What it is*: Core utility and control-flow activities used to manage pipeline execution.
   * *Key Examples*: *Wait*, *Set Variable*, *Web activity* (to call REST APIs), and *Append variable*.

8. **HDInsight**:
   * *What it is*: Activities used to execute legacy Hadoop ecosystem jobs (Hive, Pig, MapReduce, or Spark) on Azure HDInsight clusters.

9. **Iteration & Conditionals**:
   * *What it is*: Control flow logic used to build smart, dynamic workflows.
   * *Key Examples*: **ForEach** (for looping through lists of files or tables), **Until** (looping until a condition is met), and **If Condition** (branching logic based on success or failure).

10. **Machine Learning**:
    * *What it is*: Activities used to execute trained machine learning models or scoring experiments hosted in Azure Machine Learning.

11. **Power Query**:
    * *What it is*: An activity that executes visual, code-free data wrangling scripts built using the Power Query interface inside the pipeline.

### What is the Copy Activity?
* **Definition**: The **Copy Activity** is the core, serverless data movement engine inside Azure Data Factory used to copy data securely and at scale between a source and a destination (sink).
* **Simple Explanation**: Think of the Copy Activity as the **"heavy-duty moving truck"** of ADF. While pipelines handle the scheduling and workflow logic, the Copy Activity does the actual heavy lifting of reading data from your source, handling format conversions, and writing it into your destination.

### What is the Get Metadata Activity?
* **Definition**: The **Get Metadata** activity is a control-flow activity in Azure Data Factory used to retrieve metadata (information *about* data, rather than the data itself) from any supported data store or file.
* **Simple Explanation**: Think of Get Metadata as a **"scout or detective"** in your pipeline. Before your pipeline attempts to copy or transform a file or table, this activity checks things like:
  * Does the file or table actually exist?
  * How big is the file (in bytes)?
  * How many columns or rows does it have?
  * What is the structure (schema) or last modified timestamp?

### ⚙️ How to Configure Get Metadata in ADF
1. **Add Activity**: In the Author tab, expand the **General** category in the activities panel and drag **Get metadata** onto your pipeline canvas.
2. **Dataset Tab**: 
   * Select an existing dataset (or create a new one pointing to your file or table) that you want to inspect.
3. **Field List Tab**: 
   * Click **+ New** to select the specific metadata properties you want to extract. Common options include:
    * **Child items**: Returns a list of all files and subfolders located inside a specified container or directory.
    * **Exists**: Returns a boolean (`true` or `false`) indicating whether the specified file, table, or folder actually exists.
    * **Item name**: Returns the name of the file or folder being inspected.
    * **Item type**: Specifies whether the inspected item is a File or a Folder (Container).
    * **Last modified**: Returns a timestamp showing the exact date and time the file or folder was last updated.
    * **Add dynamic content**: An expression builder tool used to write code referencing outputs and variables from previous activities.
4. **Using the Output**: 
   * You can reference the output of a Get Metadata activity in subsequent activities using dynamic content expressions (e.g., `@activity('GetMetadata1').output.exists` or `@activity('GetMetadata1').output.childItems`).

### What is the If Condition Activity?
* **Definition**: The **If Condition** activity is a control-flow activity in Azure Data Factory that evaluates a boolean expression (yielding `true` or `false`) to determine which set of downstream activities should be executed.
* **Simple Explanation**: Think of the If Condition activity as a **"fork in the road"** or a traffic cop in your pipeline. It looks at a condition (like checking if a file exists or if a row count is greater than zero) and directs the pipeline down one of two paths:
  * **True Path**: Executes if the evaluated expression is true.
  * **False Path**: Executes if the evaluated expression is false.

### ⚙️ How to Configure If Condition in ADF
1. **Add Activity**: In the Author tab, expand the **Iteration & conditionals** category in the activities panel and drag **If Condition** onto your pipeline canvas.
2. **Settings Tab (The Expression)**: 
   * Click into the **Expression** box and click **Add dynamic content** to write the logical condition (e.g., `@activity('GetMetadata1').output.exists` or `@equals(activity('GetMetadata1').output.itemType, 'Folder')`).
3. **Activities Tab (True / False Blocks)**: 
   * Inside the If Condition activity properties, click on the pencil icon for the **True** activities block to drop in the steps that should run if the expression evaluates to true.
   * Do the same for the **False** activities block to build out the alternative execution path.

### What is the ForEach Activity?
* **Definition**: The **ForEach** activity is a control-flow activity in Azure Data Factory used to execute a set of activities repeatedly in a loop over a collection of items (such as a list of file names, table names, or folder paths).
* **Simple Explanation**: Think of the ForEach activity as a **"conveyor belt or automated loop"** in your pipeline. Instead of manually writing a copy activity for every single file or table, you feed it a list (like the output of a Get Metadata `childItems` list), and it loops through them one by one (or concurrently) to perform the same action on each item.

### ⚙️ How to Configure ForEach in ADF
1. **Add Activity**: In the Author tab, expand the **Iteration & conditionals** category in the activities panel and drag **For each** onto your pipeline canvas.
2. **Settings Tab**:
   * **Batch count**: Controls how many items are processed simultaneously (leave unchecked for default sequential processing, or set a number for parallel execution up to 50).
   * **Items**: Click into the box, open **Add dynamic content**, and provide the array or collection you want to loop through (e.g., pointing to the output of a Get Metadata activity: `@activity('GetMetadata1').output.childItems`).
3. **Activities Tab (Inside the Loop)**:
   * Click on the pencil icon inside the ForEach activity box to open its internal canvas.
   * Drag and drop the activities you want repeated for each item (such as a **Copy Activity**).
   * Inside those nested activities, use the loop item variable—`@item().name`—to dynamically reference the current file or table being processed in that iteration.

### What is the Data Flow Activity?
* **Definition**: A Mapping Data Flow is a visually designed data transformation execution logic in Azure Data Factory that runs on an **Apache Spark cluster** managed entirely by Azure (serverless/no-code Spark).
* **Simple Explanation**: Think of a Mapping Data Flow as the *recipe or blueprint* (the transformation logic), while the **Data Flow Activity** is the *cook or execution button* placed inside your pipeline that actually runs that recipe using an Azure-managed Spark cluster.

### ⚙️ How to Add and Configure a Data Flow Activity
1. **Create Pipeline**: Open your pipeline canvas in the Author tab.
2. **Add Activity**: Expand the **Move & transform** category in the activities panel and drag the **Data flow** activity onto your pipeline canvas.
3. **Configure Settings**: 
   * Click on the activity, go to the **Settings** tab, and select your target Data Flow from the dropdown.
   * Choose your Integration Runtime compute properties and configure any parameters if your data flow requires them.
4. **Execute**: Connect it to upstream activities (like a *Get Metadata* or *Wait* activity) or run a **Debug** session to test the Spark-powered transformation live.

## 🔀 Azure Data Flow: Multiple Inputs, Outputs & Combining Transformations

### What are Multiple Inputs and Outputs in Data Flows?
* **Multiple Inputs**: Bringing data from two or more different source streams (e.g., Source A and Source B) into a single transformation block to combine or compare them.
* **Multiple Outputs**: Taking a single incoming data stream and splitting it into two or more distinct outgoing streams based on conditions or routing rules.

### 📋 Step-by-Step Guide: Working with Multi-Stream Transformations in the Interface

#### 1. Join (Combining Two Sources)
* **What it does**: Merges two separate data streams into a single stream based on matching column keys (similar to an SQL `JOIN`).
* **Interface Steps**:
  1. Have two separate source streams (e.g., `SourceSales` and `SourceCustomer`).
  2. Click the `+` icon on your primary stream and select **Join**.
  3. In the **Join settings** tab, select your **Left stream** and **Right stream**.
  4. Choose the **Join type** (Inner, Left outer, Right outer, Full outer).
  5. Under **Join conditions**, match the common keys (e.g., `CustomerID` = `CustomerID`).

#### 2. Conditional Split (Creating Multiple Outputs)
* **What it does**: Takes a single incoming stream and routes rows into multiple different output paths based on custom expressions (acts like a multi-branch `if/else` statement).
* **Interface Steps**:
  1. Click the `+` icon on your data stream and select **Conditional split**.
  2. In the **Split conditions** tab, add new output stream names and type expressions for each rule (e.g., Stream 1 condition: `Region == 'North'`, Stream 2 condition: `Region == 'South'`).
  3. Define a **Default output stream name** for rows that don't match any condition.
  4. Downstream, each output branch becomes available as a separate selectable stream name.

#### 3. Exists (Filtering Based on Existence)
* **What it does**: Checks whether rows from your primary stream exist in a secondary stream, keeping only matching (or non-matching) rows (similar to a SQL `EXISTS` or `IN` clause).
* **Interface Steps**:
  1. Click the `+` icon on your primary data stream and select **Exists**.
  2. In the **Exists settings** tab, select the **Left stream** (primary data) and **Right stream** (lookup criteria).
  3. Choose whether you want **Exists** (keep rows that match) or **Does not exist** (filter out matching rows).
  4. Define the matching key columns.

#### 4. Union (Stacking Streams Together)
* **What it does**: Combines two or more data streams with the *same schema* vertically, stacking the rows on top of each other (similar to a SQL `UNION ALL`).
* **Interface Steps**:
  1. Click the `+` icon on your primary stream and select **Union**.
  2. In the **Union settings** tab, use the dropdown to add additional incoming streams you want to merge together.
  3. Ensure column names align correctly between the streams.

#### 5. Lookup (Attaching Reference Data)
* **What it does**: Grabs reference columns from a secondary stream and attaches them to your primary data stream based on a key match (similar to a left join, but optimized for referencing lookup tables).
* **Interface Steps**:
  1. Click the `+` icon on your primary stream and select **Lookup**.
  2. In the **Lookup settings** tab, select the secondary **Primary stream** or lookup dataset.
  3. Set your matching condition keys.
  4. In subsequent transformations, you can select columns from both your main stream and the lookup stream.

### What are Schema Modifiers?
* **Definition**: Schema modifiers are a specialized group of transformations in Azure Data Flow used to reshape, rename, cast types, add calculated columns, or drop attributes as data moves through your pipeline.
* **Why we use them**: Raw data rarely arrives in the exact format your target database or data lake requires. Schema modifiers let you clean up column names, calculate new metrics on the fly, fix data types, and prepare rows for database writes.

### 1. Derived Column
* **What it does**: Creates brand-new columns or updates existing ones using expression formulas (e.g., calculating totals, combining strings, or handling nulls).
* **Interface Steps**:
  1. Click the `+` icon on your data stream and select **Derived column**.
  2. In the configuration panel, click **+ Add** -> **Add column**.
  3. Type the **Column name** you want to create or update.
  4. Click into the **Expression** box to open the expression builder and type your formula (e.g., `Price * Quantity`).

### 2. Select
* **What it does**: Renames column headers, drops unwanted columns to save memory, or reorders the column sequence.
* **Interface Steps**:
  1. Click the `+` icon and select **Select**.
  2. Review the mapped incoming and output columns.
  3. Edit the output names to rename columns, or click the delete icon next to any column to drop it.

### 3. Aggregate
* **What it does**: Groups data by specific columns and calculates metrics using functions like `sum()`, `avg()`, `count()`, or `max()`.
* **Interface Steps**:
  1. Click the `+` icon and select **Aggregate**.
  2. Go to the **Group by** tab and choose the columns you want to group your data by (e.g., `Department` or `Region`).
  3. Go to the **Aggregates** tab, click **+ Add**, name your output metric column, and write your aggregate expression (e.g., `sum(Sales)`).

### 4. Surrogate Key
* **What it does**: Generates a sequential, auto-incrementing integer value (like 1, 2, 3...) for each row coming through the data stream, often used to create a primary key for a dimension table.
* **Interface Steps**:
  1. Click the `+` icon and select **Surrogate key**.
  2. In the configuration panel, enter a **Key column** name (e.g., `EmployeeKey`).
  3. (Optional) Set a starting value under **Start value** (default is 1).

### 5. Pivot
* **What it does**: Reshapes data by turning unique row values into new column headers (row-to-column transformation).
* **Interface Steps**:
  1. Click the `+` icon and select **Pivot**.
  2. Under **Group by**, select the columns you want to keep intact.
  3. Under **Pivot key**, select the column whose row values will become your new column headers.
  4. Under **Pivoted stacked column**, define the aggregation expression (e.g., `sum(Amount)`) that fills in the values under the new headers.

### 6. Unpivot
* **What it does**: Turns columns into rows (the exact opposite of Pivot), useful for normalizing wide tables into a long format.
* **Interface Steps**:
  1. Click the `+` icon and select **Unpivot**.
  2. Under **Unpivot key**, select the columns you want to convert into row values.
  3. Under **Group by**, select columns you want to keep as reference context.
  4. Define output column names for the unpivoted key and its corresponding value.

### 7. Window
* **What it does**: Performs calculations across a moving group of rows (such as running totals or moving averages) *without* collapsing your dataset like Aggregate does.
* **Interface Steps**:
  1. Click the `+` icon and select **Window**.
  2. Under the **Over** tab, choose the partitioning columns (grouping).
  3. Under the **Sort** tab, specify the ordering column and direction.
  4. Under the **Window columns** tab, add your calculation expressions using window functions (e.g., `sum(Revenue)` with an unbounded preceding frame).

### 8. Rank
* **What it does**: Assigns a sequential ranking number (like `dense_rank()`, `rank()`, or row numbering) to rows partitioned by specific column values.
* **Interface Steps**:
  1. Click the `+` icon and select **Rank**.
  2. In the configuration panel, name your **New column name** (e.g., `SalesRank`).
  3. Click the expression box to choose the sorting logic and ranking type (e.g., descending order by sales amount).

### 9. External Call
* **What it does**: Calls an external web service or REST API endpoint for each row passing through your data stream, enriching your data with external API responses.
* **Interface Steps**:
  1. Click the `+` icon and select **External call**.
  2. Connect or configure a Linked Service pointing to your target REST API / Web service.
  3. Map input columns from your data stream to the API request parameters, and define how the JSON response columns map back into your data flow stream.

### 10. Cast
* **What it does**: Explicitly converts the data type of a column (e.g., turning a string column into an Integer, Decimal, or Date format).
* **Interface Steps**:
  1. Click the `+` icon and select **Cast**.
  2. In the configuration panel, click **+ Add**.
  3. Select the target **Column** and choose its new data **Type** (e.g., Integer, Date, String).

### Formatters (Flatten, Parse, Stringify)
Formatters are specialized transformations used to handle complex nested data structures like JSON, arrays, and strings.

#### Flatten
* **What it does**: Takes nested array structures (common in JSON files or API responses) and flattens them out into individual, separate rows.
* **Interface Steps**:
  1. Click the `+` icon on your data stream and select **Flatten**.
  2. In the **Flatten settings** tab, select the incoming array column you want to unpack under **Unroll by**.
  3. Map the outer attributes and inner array elements to your output projection columns so each item gets its own row.

#### Parse
* **What it does**: Takes a string column containing structured text (such as an embedded JSON string or XML) and parses it into actual columns and complex data types.
* **Interface Steps**:
  1. Click the `+` icon and select **Parse**.
  2. Select the **Column to parse** (e.g., a string column holding JSON data).
  3. Choose the **Expression type** (e.g., JSON) and define the output schema structure so ADF knows how to unpack the fields.

#### Stringify
* **What it does**: The opposite of Parse; takes complex data types or nested columns and converts them into a single string (often used to serialize data into JSON format before writing it out).
* **Interface Steps**:
  1. Click the `+` icon and select **Stringify**.
  2. Select the complex column or structure you want to convert.
  3. Choose the target output format (e.g., JSON) to generate a serialized string column.

### Row Modifiers (Filter, Sort, Alter Row, Assert)
Row modifiers change *which* rows flow through your pipeline or how rows are flagged for downstream databases.

#### Filter
* **What it does**: Restricts rows based on a boolean expression condition, passing only rows that evaluate to `true` (e.g., keeping only active records).
* **Interface Steps**:
  1. Click the `+` icon and select **Filter**.
  2. Click into the **Filter on** expression box and write your logical condition (e.g., `Status == 'Active' && Sales > 1000`).

#### Sort
* **What it does**: Orders your data rows ascending or descending based on one or more columns.
* **Interface Steps**:
  1. Click the `+` icon and select **Sort**.
  2. Under the **Sort settings** tab, click **+ Add**.
  3. Select the column name and choose whether you want **Ascending** or **Descending** order.

#### Alter Row
* **What it does**: Tags rows with database operations—such as **Insert**, **Update**, **Delete**, or **Upsert**—before writing to a database sink.
* **Interface Steps**:
  1. Click the `+` icon and select **Alter row**.
  2. Click **+ Add row condition** and write expressions for each rule (e.g., set condition `isUpdate == true` and select the action **Update if**).
  3. *Note*: Ensure your database sink settings are also configured to allow inserts/updates and have key columns defined.

#### Assert
* **What it does**: Validates data quality rules inline (such as checking for nulls, range limits, or regex patterns) and lets you tag, fail, or split invalid rows.
* **Interface Steps**:
  1. Click the `+` icon and select **Assert**.
  2. Click **+ Add condition**, give your assertion a name, and write the expression rule (e.g., `Age >= 18`).
  3. Choose the failure handling action (e.g., send failed rows to an error stream or fail the data flow).

### Flowlets
* **What it does**: A reusable, modular block of data flow logic (containing a combination of transformations) that you can package once and reuse across multiple different data flows.
* **Interface Steps**:
  1. In the Author hub, click the **+** (plus) icon and select **Flowlet**.
  2. Build your reusable logic stream inside the Flowlet canvas (e.g., a standard data-cleaning sequence).
  3. In your main Data Flow, click `+`, select **Flowlet**, and drop your custom modular component directly into the pipeline stream.

### 4. Destination: Sink
* **What it does**: The final endpoint of your data flow; writes your transformed, cleaned data out to storage containers, data lakes, or databases.
* **Interface Steps**:
  1. Click the `+` icon at the very end of your data flow stream and select **Sink**.
  2. In the **Sink settings** tab, select your destination dataset (or create a new one pointing to your target container like `reporting` or an SQL table).
  3. Configure options under **Settings** (such as file naming rules, output options, or database write methods like insert/upsert).
  4. Use the **Mapping** tab to verify or adjust how incoming columns map directly to your destination schema.